In [4]:
import pandas as pd
import requests

In [5]:
SOURCE_URL = "https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_market_tracker/zip_code_market_tracker.tsv000.gz"

# Read only the header row without loading the data
df_header = pd.read_csv(SOURCE_URL, sep="\t", compression="gzip", nrows=0)

# Display all available columns and total count
print(f"Total columns available: {len(df_header.columns)}\n")
for idx, col in enumerate(df_header.columns, 1):
    print(f"{idx:2d}. {col}")

Total columns available: 58

 1. PERIOD_BEGIN
 2. PERIOD_END
 3. PERIOD_DURATION
 4. REGION_TYPE
 5. REGION_TYPE_ID
 6. TABLE_ID
 7. IS_SEASONALLY_ADJUSTED
 8. REGION
 9. CITY
10. STATE
11. STATE_CODE
12. PROPERTY_TYPE
13. PROPERTY_TYPE_ID
14. MEDIAN_SALE_PRICE
15. MEDIAN_SALE_PRICE_MOM
16. MEDIAN_SALE_PRICE_YOY
17. MEDIAN_LIST_PRICE
18. MEDIAN_LIST_PRICE_MOM
19. MEDIAN_LIST_PRICE_YOY
20. MEDIAN_PPSF
21. MEDIAN_PPSF_MOM
22. MEDIAN_PPSF_YOY
23. MEDIAN_LIST_PPSF
24. MEDIAN_LIST_PPSF_MOM
25. MEDIAN_LIST_PPSF_YOY
26. HOMES_SOLD
27. HOMES_SOLD_MOM
28. HOMES_SOLD_YOY
29. PENDING_SALES
30. PENDING_SALES_MOM
31. PENDING_SALES_YOY
32. NEW_LISTINGS
33. NEW_LISTINGS_MOM
34. NEW_LISTINGS_YOY
35. INVENTORY
36. INVENTORY_MOM
37. INVENTORY_YOY
38. MONTHS_OF_SUPPLY
39. MONTHS_OF_SUPPLY_MOM
40. MONTHS_OF_SUPPLY_YOY
41. MEDIAN_DOM
42. MEDIAN_DOM_MOM
43. MEDIAN_DOM_YOY
44. AVG_SALE_TO_LIST
45. AVG_SALE_TO_LIST_MOM
46. AVG_SALE_TO_LIST_YOY
47. SOLD_ABOVE_LIST
48. SOLD_ABOVE_LIST_MOM
49. SOLD_ABOVE_LIST_YO

In [10]:
SOURCE_URL = "https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_market_tracker/zip_code_market_tracker.tsv000.gz"
OUTPUT_FILE = "redfin_ma_market_metrics_2023_2026.csv"

START_DATE = "2023-01-01"
END_DATE = "2026-09-30"

# Target columns + metadata needed for indexing and filtering
TARGET_COLUMNS = [
    # Metadata
    "PERIOD_BEGIN",
    "PERIOD_END",
    "REGION",
    "STATE_CODE",
    "PROPERTY_TYPE",

    # Requested Market Metrics
    "MEDIAN_SALE_PRICE",
    "MEDIAN_LIST_PRICE",
    "MEDIAN_PPSF",
    "MEDIAN_LIST_PPSF",
    "HOMES_SOLD",
    "PENDING_SALES",
    "NEW_LISTINGS",
    "INVENTORY",
    "MONTHS_OF_SUPPLY",
    "MEDIAN_DOM",
    "AVG_SALE_TO_LIST",
    "SOLD_ABOVE_LIST",
    "PRICE_DROPS",
    "OFF_MARKET_IN_TWO_WEEKS"
]

print("Streaming and filtering dataset from Redfin S3...")

chunk_size = 100_000
ma_records = []
total_rows = 0

for chunk in pd.read_csv(
    SOURCE_URL,
    sep="\t",
    compression="gzip",
    usecols=lambda col: col in TARGET_COLUMNS,
    chunksize=chunk_size,
    low_memory=False
):
    total_rows += len(chunk)

    # 1. State filter
    ma_chunk = chunk[chunk["STATE_CODE"].astype(str).str.upper() == "MA"].copy()

    if not ma_chunk.empty:
        # 2. Date range filter
        ma_chunk["PERIOD_BEGIN"] = pd.to_datetime(ma_chunk["PERIOD_BEGIN"])
        mask = (ma_chunk["PERIOD_BEGIN"] >= START_DATE) & (ma_chunk["PERIOD_BEGIN"] <= END_DATE)
        filtered = ma_chunk[mask]

        if not filtered.empty:
            ma_records.append(filtered)

    print(f"Scanned {total_rows:,} rows... Found {sum(len(c) for c in ma_records):,} MA matches.", end="\r")

print("\nCombining results and exporting...")

if ma_records:
    df_ma = pd.concat(ma_records, ignore_index=True)

    print(f"Done! Saved {len(df_ma):,} records to '{OUTPUT_FILE}'.")
else:
    print("No records matched the criteria.")

Streaming and filtering dataset from Redfin S3...
Scanned 9,725,026 rows... Found 70,176 MA matches.
Combining results and exporting...
Done! Saved 70,176 records to 'redfin_ma_market_metrics_2023_2026.csv'.


In [11]:
df_ma

,PERIOD_BEGIN,PERIOD_END,REGION,STATE_CODE,PROPERTY_TYPE,MEDIAN_SALE_PRICE,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_LIST_PPSF,HOMES_SOLD,PENDING_SALES,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,AVG_SALE_TO_LIST,SOLD_ABOVE_LIST,PRICE_DROPS,OFF_MARKET_IN_TWO_WEEKS
0,2025-05-01,2025-07-31,Zip Code: 01240,MA,Condo/Co-op,355500.0,612500.0,283.682635,353.392857,20.0,17.0,20.0,9.0,NaN,34.0,0.977316,0.250000,NaN,0.352941
1,2024-10-01,2024-12-31,Zip Code: 02136,MA,All Residential,695000.0,720250.0,395.377129,397.656444,65.0,57.0,54.0,17.0,NaN,21.0,1.025816,0.615385,NaN,0.684211
2,2024-08-01,2024-10-31,Zip Code: 02053,MA,Condo/Co-op,450000.0,394950.0,393.586006,433.905945,3.0,4.0,4.0,1.0,NaN,13.0,1.026667,0.333333,NaN,1.000000
3,2025-10-01,2025-12-31,Zip Code: 02134,MA,Condo/Co-op,399000.0,554000.0,652.173913,664.349898,7.0,12.0,20.0,14.0,NaN,38.0,0.954771,0.000000,NaN,0.000000
4,2024-10-01,2024-12-31,Zip Code: 01908,MA,Multi-Family (2-4 Unit),1550000.0,NaN,386.630082,NaN,1.0,NaN,NaN,NaN,NaN,123.0,0.968750,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70171,2023-06-01,2023-08-31,Zip Code: 01543,MA,Single Family Residential,553750.0,435000.0,270.898935,249.972222,24.0,25.0,29.0,10.0,NaN,18.0,1.037390,0.708333,NaN,0.800000
70172,2024-01-01,2024-03-31,Zip Code: 01770,MA,Single Family Residential,1100000.0,1489500.0,363.196126,380.858614,9.0,12.0,16.0,7.0,NaN,48.0,0.979155,0.222222,NaN,0.583333
70173,2023-10-01,2023-12-31,Zip Code: 01851,MA,Townhouse,356000.0,339499.5,294.768913,281.355659,6.0,4.0,4.0,2.0,NaN,18.0,1.048688,0.833333,NaN,0.750000
70174,2025-02-01,2025-04-30,Zip Code: 01255,MA,Single Family Residential,1317500.0,575000.0,471.789100,294.527897,2.0,2.0,9.0,10.0,NaN,129.5,0.965909,0.000000,NaN,0.000000


In [14]:
from google.colab import files

# 1. Convert the DataFrame to a CSV file
df_ma.to_csv('redfin_data.csv', index=False)

# 2. Download the CSV file to your local machine
files.download('redfin_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
ar_df = df_ma[df_ma['PROPERTY_TYPE'] == 'All Residential']
ar_df.head()

,PERIOD_BEGIN,PERIOD_END,REGION,STATE_CODE,PROPERTY_TYPE,MEDIAN_SALE_PRICE,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_LIST_PPSF,HOMES_SOLD,PENDING_SALES,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,AVG_SALE_TO_LIST,SOLD_ABOVE_LIST,PRICE_DROPS,OFF_MARKET_IN_TWO_WEEKS
1,2024-10-01,2024-12-31,Zip Code: 02136,MA,All Residential,695000.0,720250.0,395.377129,397.656444,65.0,57.0,54.0,17.0,NaN,21.0,1.025816,0.615385,NaN,0.684211
5,2025-08-01,2025-10-31,Zip Code: 01982,MA,All Residential,887500.0,849900.0,405.535340,393.372703,24.0,20.0,21.0,13.0,NaN,22.5,1.013580,0.583333,NaN,0.350000
11,2025-04-01,2025-06-30,Zip Code: 02333,MA,All Residential,575000.0,567500.0,340.000000,334.626667,37.0,42.0,54.0,21.0,NaN,20.0,1.009506,0.540541,NaN,0.619048
15,2026-01-01,2026-03-31,Zip Code: 02302,MA,All Residential,468500.0,477400.0,339.830450,363.996082,58.0,73.0,74.0,42.0,NaN,42.5,0.993103,0.465517,NaN,0.589041
16,2023-10-01,2023-12-31,Zip Code: 01013,MA,All Residential,287000.0,277450.0,199.652778,191.319318,49.0,48.0,50.0,18.0,NaN,21.0,1.023278,0.591837,NaN,0.687500


In [21]:
ar_df.isna().sum()

,0
PERIOD_BEGIN,0
PERIOD_END,0
REGION,0
STATE_CODE,0
PROPERTY_TYPE,0
MEDIAN_SALE_PRICE,0
MEDIAN_LIST_PRICE,468
MEDIAN_PPSF,1
MEDIAN_LIST_PPSF,468
HOMES_SOLD,0


In [22]:
# Extract the unique zip codes from the REGION column
# The region format is typically 'Zip Code: XXXXX'
unique_zips = df_ma['REGION'].str.replace('Zip Code: ', '', regex=False).dropna().unique()

# Sort them alphabetically/numerically
unique_zips = sorted(unique_zips)

print(f"Total unique zip codes found: {len(unique_zips)}")
print("First 20 unique zip codes:", unique_zips[:20])

Total unique zip codes found: 543
First 20 unique zip codes: ['01001', '01002', '01004', '01005', '01007', '01008', '01009', '01010', '01011', '01012', '01013', '01020', '01022', '01026', '01027', '01028', '01029', '01030', '01031', '01032']


In [23]:
unique_zips

['01001',
 '01002',
 '01004',
 '01005',
 '01007',
 '01008',
 '01009',
 '01010',
 '01011',
 '01012',
 '01013',
 '01020',
 '01022',
 '01026',
 '01027',
 '01028',
 '01029',
 '01030',
 '01031',
 '01032',
 '01033',
 '01034',
 '01035',
 '01036',
 '01037',
 '01038',
 '01039',
 '01040',
 '01050',
 '01053',
 '01054',
 '01056',
 '01057',
 '01060',
 '01062',
 '01066',
 '01068',
 '01069',
 '01070',
 '01071',
 '01072',
 '01073',
 '01074',
 '01075',
 '01077',
 '01079',
 '01080',
 '01081',
 '01082',
 '01083',
 '01084',
 '01085',
 '01088',
 '01089',
 '01092',
 '01093',
 '01095',
 '01096',
 '01097',
 '01098',
 '01103',
 '01104',
 '01105',
 '01106',
 '01107',
 '01108',
 '01109',
 '01115',
 '01116',
 '01118',
 '01119',
 '01128',
 '01129',
 '01138',
 '01139',
 '01151',
 '01201',
 '01220',
 '01222',
 '01223',
 '01224',
 '01225',
 '01226',
 '01227',
 '01229',
 '01230',
 '01235',
 '01236',
 '01237',
 '01238',
 '01240',
 '01242',
 '01243',
 '01245',
 '01247',
 '01253',
 '01254',
 '01255',
 '01256',
 '01257',


In [25]:
duplicates = ar_df[ar_df.duplicated()]
duplicates

,PERIOD_BEGIN,PERIOD_END,REGION,STATE_CODE,PROPERTY_TYPE,MEDIAN_SALE_PRICE,MEDIAN_LIST_PRICE,MEDIAN_PPSF,MEDIAN_LIST_PPSF,HOMES_SOLD,PENDING_SALES,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,AVG_SALE_TO_LIST,SOLD_ABOVE_LIST,PRICE_DROPS,OFF_MARKET_IN_TWO_WEEKS
